# Humanitarian News Classification Notebook

The Jupyter notebook pipeline demonstrates a pragmatic approach to collecting and structuring humanitarian news data. It balances simplicity, reproducibility, and relevance to MSF operations while remaining transparent about its assumptions and limitations.

## Key Sections
1. **Data Processing**: Loading, analysing and preprocessing the dataset for classification models.
2. **Topic Modelling**: Applying classification model for topic modelling.
3. **Sentiment Classification**: Applying classification model for sentiment analysis.
4. **Output**: Generating structured output.

Repository: [JP-Thoma/Humanitarian-News-Classification](https://github.com/JP-Thoma/Humanitarian-News-Classification)

## 1. Data Processing: Loading, analysing and preprocessing the dataset for classification models

In [17]:
# installing dependencies
!pip install -r requirements.txt

In [ ]:
# load libraries
import os
from dotenv import load_dotenv
from newsapi import NewsApiClient
import pandas as pd
from transformers import pipeline

In [ ]:
# load api key and HuggingFace login
load_dotenv()

API_KEY = os.getenv("NEWSAPI_KEY")

if API_KEY is None:
    raise ValueError("API key not found. Please check your .env file.")

HfHubHTTPError: You've hit the rate limit for the /whoami-v2 endpoint, which is intentionally strict for security reasons. If you're calling it often, consider caching the response with `whoami(..., cache=True)`.

In [22]:
login(os.getenv("HUGGINGFACE_TOKEN"))

HfHubHTTPError: You've hit the rate limit for the /whoami-v2 endpoint, which is intentionally strict for security reasons. If you're calling it often, consider caching the response with `whoami(..., cache=True)`.

In [7]:
# intitialise NewsAPI Client
newsapi = NewsApiClient(api_key=API_KEY)

In [8]:
# specify keywords
QUERY = """
(
  outbreak OR epidemic OR pandemic OR cholera OR measles OR malaria OR dengue OR tuberculosis OR Ebola OR diphtheria OR malnutrition OR starvation OR famine OR "maternal mortality" OR trauma OR "mass casualty" OR "burn injuries" 
  OR "health system" OR "hospital overwhelmed"
)
AND
(
  war OR conflict OR violence OR bombardment OR airstrike OR siege OR displacement OR refugee OR IDP OR famine OR "food insecurity"
)
AND
(
  humanitarian OR NGO OR "aid workers" OR "medical charity"
)
"""

In [9]:
# specify country selection
MSF_COUNTRIES = [
    # Africa
    "Benin", "Burkina Faso", "Burundi", "Cameroon", "Central African Republic",
    "Chad", "Comoros", "Democratic Republic of Congo", "Eswatini", "Ethiopia",
    "Guinea", "Guinea-Bissau", "Ivory Coast", "Kenya", "Liberia",
    "Madagascar", "Malawi", "Mali", "Mauritania", "Mozambique",
    "Niger", "Nigeria", "Sierra Leone", "Somalia", "Somaliland",
    "South Africa", "South Sudan", "Sudan", "Tanzania", "Uganda", "Zimbabwe",

    # Middle East & North Africa
    "Egypt", "Iran", "Iraq", "Jordan", "Lebanon",
    "Libya", "Palestine", "Syria", "Yemen",

    # Asia & Pacific
    "Bangladesh", "Cambodia", "North Korea", "Hong Kong", "India",
    "Indonesia", "Kiribati", "Malaysia", "Myanmar", "Pakistan",
    "Papua New Guinea", "Philippines", "Thailand",

    # Europe & Central Asia
    "Afghanistan", "Armenia", "Azerbaijan", "Belarus", "Belgium",
    "Bulgaria", "France", "Greece", "Italy", "Kazakhstan",
    "Kyrgyzstan", "Latvia", "Lithuania", "Poland",
    "Russia", "Serbia", "Tajikistan", "Turkey",
    "Ukraine", "United Kingdom", "Uzbekistan",

    # Americas
    "Brazil", "Colombia", "El Salvador", "Guatemala",
    "Haiti", "Honduras", "Jamaica", "Mexico",
    "Nicaragua", "Panama", "United States", "Venezuela"
]

In [10]:
# fetch articles
articles_response = newsapi.get_everything(
    q=QUERY,
    from_param="2026-04-15",
    to="2026-04-25",
    language="en",
    sort_by="publishedAt",
    page_size=100
)

print(f"Total results: {articles_response['totalResults']}")
print(f"Articles retrieved: {len(articles_response['articles'])}")

articles_response['articles'][0]

Total results: 176
Articles retrieved: 91


{'source': {'id': None, 'name': 'Lse.ac.uk'},
 'author': 'Blog Admin',
 'title': 'What it means if US actions during the Iran conflict may have broken international humanitarian law',
 'description': 'Since the conflict began, the US has faced accusations that some of its actions against Iran may constitute violations of international law. Jim Rice writes that these alleged breaches should be seen in the context of previous US actions targeting suspected d…',
 'url': 'https://blogs.lse.ac.uk/usappblog/2026/04/24/what-it-means-if-us-actions-during-the-iran-conflict-may-have-broken-international-humanitarian-law/',
 'urlToImage': 'https://blogsmedia.lse.ac.uk/blogs.dir/58/files/2026/04/No-war-with-Iran-sign-Flickr-2000x1250-1.jpg',
 'publishedAt': '2026-04-24T17:33:45Z',
 'content': 'Since the conflict began, the US has faced accusations that some of its actions against Iran may constitute violations of international law. Jim Rice writes that these alleged breaches should be seen… [+7459

In [11]:
# create dataframe
articles = articles_response['articles']

df = pd.json_normalize(articles)
df.head()

,author,title,description,url,urlToImage,publishedAt,content,source.id,source.name
0,Blog Admin,What it means if US actions during the Iran co...,"Since the conflict began, the US has faced acc...",https://blogs.lse.ac.uk/usappblog/2026/04/24/w...,https://blogsmedia.lse.ac.uk/blogs.dir/58/file...,2026-04-24T17:33:45Z,"Since the conflict began, the US has faced acc...",NaN,Lse.ac.uk
1,Paul Morgan,Middle East Reconstruction Poses a Carbon Shoc...,New academic research suggests reconstruction ...,https://gcaptain.com/middle-east-reconstructio...,https://gcaptain.com/wp-content/uploads/2017/1...,2026-04-24T17:20:52Z,New academic research suggests reconstruction ...,NaN,gcaptain.com
2,Nour Abo Aisha,"Lacking Proper Sanitation, Gaza’s Tent Camps A...","As conditions in Gaza’s tent camps worsen, rod...",https://truthout.org/articles/lacking-proper-s...,https://truthout.org/app/uploads/2026/04/Getty...,2026-04-24T17:04:09Z,Part of the Series\r\nTruthout is a vital news...,NaN,Truthout
3,NaN,Trump’s War of Choice With Iran Threatens a Gl...,President Trump’s war of choice is generating ...,https://freerepublic.com/focus/f-news/4376287/...,NaN,2026-04-24T17:00:29Z,Skip to comments.\r\nTrumps War of Choice With...,NaN,Freerepublic.com
4,Muskan Singh,Quote of the Day by Michael Jackson: 'The most...,"Quote of the Day: Michael Jackson, the 'King o...",https://economictimes.indiatimes.com/news/inte...,"https://img.etimg.com/thumb/msid-130499018,wid...",2026-04-24T16:14:57Z,Quote of the Day: A powerful Quote of the Day ...,the-times-of-india,The Times of India


In [12]:
# select relevant columns
df = df[[
    "source.name",
    "author",
    "title",
    "description",
    "url",
    "publishedAt",
    "content"
]]

df.rename(columns={
    "source.name": "source"
}, inplace=True)

print(f"Final dataset size: {len(df)}")
df.head()

Final dataset size: 91


,source,author,title,description,url,publishedAt,content
0,Lse.ac.uk,Blog Admin,What it means if US actions during the Iran co...,"Since the conflict began, the US has faced acc...",https://blogs.lse.ac.uk/usappblog/2026/04/24/w...,2026-04-24T17:33:45Z,"Since the conflict began, the US has faced acc..."
1,gcaptain.com,Paul Morgan,Middle East Reconstruction Poses a Carbon Shoc...,New academic research suggests reconstruction ...,https://gcaptain.com/middle-east-reconstructio...,2026-04-24T17:20:52Z,New academic research suggests reconstruction ...
2,Truthout,Nour Abo Aisha,"Lacking Proper Sanitation, Gaza’s Tent Camps A...","As conditions in Gaza’s tent camps worsen, rod...",https://truthout.org/articles/lacking-proper-s...,2026-04-24T17:04:09Z,Part of the Series\r\nTruthout is a vital news...
3,Freerepublic.com,NaN,Trump’s War of Choice With Iran Threatens a Gl...,President Trump’s war of choice is generating ...,https://freerepublic.com/focus/f-news/4376287/...,2026-04-24T17:00:29Z,Skip to comments.\r\nTrumps War of Choice With...
4,The Times of India,Muskan Singh,Quote of the Day by Michael Jackson: 'The most...,"Quote of the Day: Michael Jackson, the 'King o...",https://economictimes.indiatimes.com/news/inte...,2026-04-24T16:14:57Z,Quote of the Day: A powerful Quote of the Day ...


In [13]:
# light cleaning
# Drop duplicates based on title
df.drop_duplicates(subset="title", inplace=True)

# Drop rows with no description (important for later classification)
df = df[df["description"].notna()]

df.reset_index(drop=True, inplace=True)

print(f"Final dataset size: {len(df)}")

Final dataset size: 90


## 2. Topic Modelling: Applying classification model for topic modelling

In [1]:
# load topic modelling pipeline from Hugging Face
pipe_topics = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

NameError: name 'pipeline' is not defined

In [ ]:
sequence_to_classify = "one day I will see the world"
candidate_labels = ['travel', 'cooking', 'dancing']
pipe_topics(sequence_to_classify, candidate_labels)

In [ ]:
labels = [
    "Conflict-Driven Crisis (war, violence, attacks on healthcare)",
    "Epidemic & Disease-Driven Crisis (cholera, outbreak, infectious disease)",
    "Food & Nutrition-Driven Crisis (famine, malnutrition, starvation)",
    "Displacement-Driven Crisis (refugees, camps, migration)",
    "Environmental Disaster-Driven Crisis (flood, drought, earthquake)"
]

## 3. Sentiment Classification: Applying classification model for sentiment analysis

In [20]:
# load sentiment analysis pipeline from Hugging Face
pipe_sentiments = pipeline("text-classification", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")

HTTP Error 429 thrown while requesting HEAD https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english/resolve/main/config.json
Retrying in 1s [Retry 1/5].


HTTP Error 429 thrown while requesting HEAD https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english/resolve/main/config.json
Retrying in 2s [Retry 2/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english/resolve/main/config.json
Retrying in 4s [Retry 3/5].


KeyboardInterrupt: 

In [21]:
pipe_sentiments = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    local_files_only=False
)

HTTP Error 429 thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english/resolve/main/config.json
Retrying in 1s [Retry 1/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english/resolve/main/config.json
Retrying in 2s [Retry 2/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english/resolve/main/config.json
Retrying in 4s [Retry 3/5].


KeyboardInterrupt: 

## 4. Output: Generating structured output